In [1]:
# conda activate dream

library(sva)
library(edgeR)
library(limma)
library(dplyr)
library(data.table)

setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex")

Loading required package: mgcv

Loading required package: nlme

This is mgcv 1.9-3. For overview type 'help("mgcv-package")'.

Loading required package: genefilter

Loading required package: BiocParallel

Loading required package: limma


Attaching package: ‘dplyr’


The following object is masked from ‘package:nlme’:

    collapse


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last




In [2]:
batch_ANOVA <- function(pca, sampleinfo, batch_cols) {
    for (i in batch_cols) {
        batch_label <- colnames(sampleinfo)[i]
        batch <- factor(sampleinfo[,i])
        if (length(levels(batch)) > 1) {
            if (batch_label == "Mean_age") {
                batch <- sampleinfo[,i] 
            }
            fit <- lm(pca$x[,2] ~ batch)
            pc_test <- anova(fit)$`Pr(>F)`[1]
            print(paste(batch_label, p.adjust(pc_test, "BH")))
        }
    }
}

### Prep data

In [3]:
sampleinfo <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/bulk/GTEx/cortex/GTEx_cortex_sampleinfo.csv", data.table=FALSE)

In [ ]:
expr <- fread("GTEx_cortex_counts_TMMF_SampleNetworks/All_02-25-12/GTEx_cortex_counts_TMMF_All_501_outliers_removed.csv", data.table=FALSE)

In [5]:
sampleinfo$Mean_age <- sapply(
    strsplit(sampleinfo$AGE, "-"), function(x) mean(as.numeric(x))
)

batch_cols <- c(
    grep("Mean_age", colnames(sampleinfo)),
    grep("SMCENTER", colnames(sampleinfo)),
    grep("SMNABTCH$", colnames(sampleinfo)), 
    grep("SMGEBTCH$", colnames(sampleinfo)),
    grep("SEX", colnames(sampleinfo)),
    grep("SMTSD", colnames(sampleinfo)),
    grep("DTHHRDY", colnames(sampleinfo))
)

In [6]:
expr[1:5,1:4]

,datExprT...c.1.skip1..,GTEX.111FC.0011.R3b.SM.GJ3PN,GTEX.117XS.0011.R3a.SM.GIN8W,GTEX.1192X.0011.R3b.SM.GIN8Y
,<chr>,<dbl>,<dbl>,<dbl>
1,DDX11L1,0.000000,0.0000,0.0000
2,WASH7P,122.349063,172.4276,130.2001
3,MIR6859-1,0.000000,0.0000,0.0000
4,MIR1302-2HG,1.699293,0.0000,0.0000
5,FAM138A,0.000000,0.0000,0.0000


In [ ]:
sampleinfo[,1] <- make.names(sampleinfo[,1])
sampleinfo <- sampleinfo[match(colnames(expr)[-1], sampleinfo[,1]),]

In [8]:
# For duplicate genes, choose row with highest mean expression

mean_expr <- data.frame(
    Index=1:nrow(expr), 
    Gene=expr[,1], 
    Expr=rowMeans(expr[,-1])
)

mean_expr <- mean_expr %>%
    group_by(Gene) %>%
    slice_max(Expr, with_ties=FALSE)

print(dim(mean_expr))

expr <- expr[mean_expr$Index,]

[1] 56027     3


In [9]:
# Remove 0 expression genes
expr_filtered <- expr[rowSums(expr[,-1]) > 0,]
dim(expr_filtered)

[1] 55405   502

In [10]:
rownames(expr_filtered) <- expr_filtered[,1]
dat <- expr_filtered[,-1]
dim(dat)

[1] 55405   501

In [11]:
dat[1:5, 1:5]

,GTEX.111FC.0011.R3b.SM.GJ3PN,GTEX.117XS.0011.R3a.SM.GIN8W,GTEX.1192X.0011.R3b.SM.GIN8Y,GTEX.11DXW.0011.R3b.SM.DNZZE,GTEX.11GS4.0011.R3b.SM.GJ3RI
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
5S_rRNA,0.0000,0.0000,3.88657,0.0000,0.0000
5_8S_rRNA,0.0000,0.0000,0.00000,0.0000,0.0000
7SK,0.0000,0.0000,0.00000,0.0000,0.0000
A1BG,342.4074,346.9579,391.57196,477.8993,302.1102
A1BG-AS1,291.4287,123.0124,188.49866,137.1188,163.9500


### Check significance of batch variables

In [12]:
# Before ComBat

pca <- prcomp(t(dat), center=TRUE, scale.=TRUE)

In [13]:
batch_ANOVA(pca, sampleinfo, batch_cols)

[1] "Mean_age 0.00416322152226536"
[1] "SMCENTER 0.570788741083115"
[1] "SMNABTCH 0.318237274386344"
[1] "SMGEBTCH 0.00944559570404607"
[1] "SEX 0.128915781325476"
[1] "SMTSD 5.80896561287791e-08"
[1] "DTHHRDY 4.78920238164803e-10"


### Run ComBat

In [14]:
mod <- model.matrix(~Mean_age + as.factor(SEX) + as.factor(SMTSD), data=sampleinfo)

batch <- factor(sampleinfo$SMGEBTCH)

expr_combat1 <- sva::ComBat(
    dat,
    batch,
    mod=mod,
    mean.only=FALSE,
    par.prior=TRUE,
    prior.plots=FALSE
)

Found 31327 genes with uniform expression within a single batch (all zeros); these will not be adjusted for batch.


Using the 'mean only' version of ComBat

Found177batches

Note: one batch has only one sample, setting mean.only=TRUE

Adjusting for4covariate(s) or covariate level(s)

Standardizing Data across genes

Fitting L/S model and finding priors

Finding parametric adjustments

Adjusting the Data




### Check significance of batch variables

In [15]:
# After ComBat

pca <- prcomp(t(expr_combat1), center=TRUE, scale.=TRUE)

batch_ANOVA(pca, sampleinfo, batch_cols)

[1] "Mean_age 0.545846860493071"
[1] "SMCENTER 2.16220172715102e-17"
[1] "SMNABTCH 0.0371195344770695"
[1] "SMGEBTCH 0.999998889512743"
[1] "SEX 0.567328474290017"
[1] "SMTSD 2.6554997273536e-21"
[1] "DTHHRDY 0.000562380555184244"


In [16]:
fwrite(expr_combat1, file=paste0("data/GTEx_cortex_counts_All_501_outliers_removed_TMM_ComBat_SMGEBTCH_corrected.csv"), row.names=TRUE)

x being coerced from class: matrix to data.table

